# Robustness Testing: Walk-Forward, VaR & Monte Carlo

This notebook stress-tests the optimal carry trade portfolio (70% JPY / 27.4% MXN reverse / 2.6% CNY reverse)
using three complementary approaches:

1. **Walk-Forward Optimization** — Does the allocation hold up out-of-sample?
2. **Value at Risk (VaR) & CVaR** — What's the worst-case loss at various confidence levels?
3. **Monte Carlo Simulation** — What does the distribution of future outcomes look like?

All three build directly on the daily return series from the previous notebooks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date, timedelta
from scipy.optimize import minimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

# --- Parameters ---
end = date.today()
start = end - timedelta(days=365*8 + 10)
start_s = start.isoformat()
end_s = end.isoformat()

def fred_csv(series_id, start_date, end_date):
    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={series_id}&cosd={start_date}&coed={end_date}"
    )
    df = pd.read_csv(url)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.set_index("observation_date")
    df = df.replace(".", np.nan)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

print(f"Data window: {start_s} to {end_s}")

## 0. Rebuild the Return Series

Reconstruct the three carry return series we need: JPY funding carry, MXN reverse carry, CNY reverse carry.

In [ ]:
# ============================================================
# PULL DATA
# ============================================================
fx_series = {
    "USDJPY": "DEXJPUS",
    "USDMXN": "DEXMXUS",
    "USDCNY": "DEXCHUS",
}

rate_series = {
    "r_us":  "EFFR",
    "r_jpy": "IRSTCI01JPM156N",
    "r_mxn": "IRSTCI01MXM156N",
    "r_cny": "IRSTCI01CNM156N",
}

fx_data = {}
for name, sid in fx_series.items():
    df = fred_csv(sid, start_s, end_s)
    df.columns = [name]
    fx_data[name] = df

rates = {}
for name, sid in rate_series.items():
    df = fred_csv(sid, start_s, end_s)
    df.columns = [name]
    rates[name] = df

fx = pd.concat(fx_data.values(), axis=1).sort_index()
rates_df = pd.concat(rates.values(), axis=1).sort_index().ffill()
merged = fx.join(rates_df, how='left').ffill().dropna()

dt = 1 / 360

# --- JPY funding carry (borrow JPY, invest USD) ---
S_jpy = merged["USDJPY"]
r_us = merged["r_us"] / 100.0
r_jpy = merged["r_jpy"] / 100.0
ret_jpy = (S_jpy.shift(-1) / S_jpy) * ((1 + r_us * dt) / (1 + r_jpy * dt)) - 1

# --- MXN reverse carry (borrow USD, invest MXN) ---
S_mxn = merged["USDMXN"]
r_mxn = merged["r_mxn"] / 100.0
ret_mxn_rev = (S_mxn / S_mxn.shift(-1)) * ((1 + r_mxn * dt) / (1 + r_us * dt)) - 1

# --- CNY reverse carry (borrow USD, invest CNY) ---
S_cny = merged["USDCNY"]
r_cny = merged["r_cny"] / 100.0
ret_cny_rev = (S_cny / S_cny.shift(-1)) * ((1 + r_cny * dt) / (1 + r_us * dt)) - 1

# Combine
returns_df = pd.DataFrame({
    "JPY": ret_jpy,
    "MXN_rev": ret_mxn_rev,
    "CNY_rev": ret_cny_rev,
}).dropna()

# Static optimal portfolio
W_OPTIMAL = np.array([0.70, 0.274, 0.026])
ret_optimal = (returns_df * W_OPTIMAL).sum(axis=1)
ret_jpy_only = returns_df["JPY"]

print(f"Period: {returns_df.index.min().date()} to {returns_df.index.max().date()}")
print(f"Days: {len(returns_df)}")
print(f"\nOptimal portfolio — Ann. ret: {ret_optimal.mean()*252:.2%}, Vol: {ret_optimal.std()*np.sqrt(252):.2%}, Sharpe: {ret_optimal.mean()/ret_optimal.std()*np.sqrt(252):.3f}")
print(f"JPY Only         — Ann. ret: {ret_jpy_only.mean()*252:.2%}, Vol: {ret_jpy_only.std()*np.sqrt(252):.2%}, Sharpe: {ret_jpy_only.mean()/ret_jpy_only.std()*np.sqrt(252):.3f}")

---
# Part 1: Walk-Forward Optimization

The in-sample optimizer used the *entire* history to find weights. Walk-forward tests whether
those weights are stable by:

1. Using only data up to time $t$ to compute optimal weights
2. Applying those weights to the *next* period (out-of-sample)
3. Sliding the window forward and repeating

If the out-of-sample Sharpe is close to 0.864, the allocation is robust. If it collapses, it was overfit.

In [ ]:
# ============================================================
# WALK-FORWARD OPTIMIZER
# ============================================================
TRAIN_DAYS = 252 * 3    # 3 years of training data
STEP_DAYS  = 126        # re-optimize every 6 months
N_ASSETS   = 3

def optimize_sharpe(returns_window):
    """Find max-Sharpe weights given a window of daily returns."""
    mu = returns_window.mean().values * 252
    cov = returns_window.cov().values * 252
    
    def neg_sharpe(w):
        r = w @ mu
        v = np.sqrt(w @ cov @ w)
        return -r / v if v > 1e-10 else 0
    
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = [(0.30, 0.80)] + [(0.0, 0.50)] * (N_ASSETS - 1)
    w0 = np.array([0.6, 0.3, 0.1])
    
    result = minimize(neg_sharpe, w0, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x if result.success else W_OPTIMAL

# Run walk-forward
wf_returns = []        # out-of-sample daily returns
wf_weights_history = [] # weight history for analysis
wf_dates = []

idx = returns_df.index
i = TRAIN_DAYS

while i + STEP_DAYS <= len(returns_df):
    # Training window
    train = returns_df.iloc[i - TRAIN_DAYS : i]
    
    # Optimize on training data
    w = optimize_sharpe(train)
    
    # Apply to next STEP_DAYS (out-of-sample)
    oos = returns_df.iloc[i : i + STEP_DAYS]
    oos_ret = (oos * w).sum(axis=1)
    
    wf_returns.append(oos_ret)
    wf_weights_history.append({
        'date': idx[i],
        'JPY': w[0], 'MXN_rev': w[1], 'CNY_rev': w[2]
    })
    
    i += STEP_DAYS

# Concatenate OOS returns
ret_wf = pd.concat(wf_returns)
weights_history = pd.DataFrame(wf_weights_history).set_index('date')

# Also compute static optimal and JPY-only over the same OOS period
oos_mask = ret_wf.index
ret_static_oos = ret_optimal.reindex(oos_mask)
ret_jpy_oos = ret_jpy_only.reindex(oos_mask)

print(f"Walk-forward OOS period: {ret_wf.index.min().date()} to {ret_wf.index.max().date()}")
print(f"Number of re-optimizations: {len(wf_weights_history)}")
print(f"\n{'Strategy':<30} {'Ann. Ret':>10} {'Ann. Vol':>10} {'Sharpe':>10} {'Max DD':>10}")
print("-" * 72)

for name, ret in [("Walk-Forward Optimal", ret_wf),
                   ("Static Optimal (in-sample)", ret_static_oos),
                   ("JPY Only", ret_jpy_oos)]:
    ret = ret.dropna()
    ar = ret.mean() * 252
    av = ret.std() * np.sqrt(252)
    sr = ar / av
    cum = (1 + ret).cumprod()
    dd = ((cum - cum.cummax()) / cum.cummax()).min()
    print(f"{name:<30} {ar:>9.2%} {av:>9.2%} {sr:>9.3f} {dd:>9.2%}")

In [ ]:
# ============================================================
# WALK-FORWARD VISUALIZATION
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(15, 13),
                         gridspec_kw={'height_ratios': [2, 1, 1]})

# Panel 1: Cumulative returns comparison
ax = axes[0]
for name, ret, color, lw in [
    ("Walk-Forward Optimal", ret_wf, '#d4a24e', 2.5),
    ("Static Optimal (in-sample)", ret_static_oos, '#7c6dd8', 1.5),
    ("JPY Only", ret_jpy_oos, '#e05252', 1.5),
]:
    ret = ret.dropna()
    cum = (1 + ret).cumprod()
    ax.plot(cum.index, cum.values, label=name, color=color, linewidth=lw)

ax.axhline(1, color='white', linewidth=0.3, linestyle='--')
ax.set_title("Walk-Forward vs Static vs JPY Only (Out-of-Sample Period)", fontsize=13)
ax.set_ylabel("Growth of $1")
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Panel 2: Rolling weight allocations over time
ax = axes[1]
ax.stackplot(weights_history.index,
             weights_history['JPY'].values,
             weights_history['MXN_rev'].values,
             weights_history['CNY_rev'].values,
             labels=['JPY', 'MXN rev', 'CNY rev'],
             colors=['#e05252', '#3ba1c9', '#9b6dd8'],
             alpha=0.8, step='post')
ax.set_title("Walk-Forward Optimal Weights Over Time", fontsize=12)
ax.set_ylabel("Weight")
ax.set_ylim(0, 1)
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Panel 3: Rolling 6M Sharpe
ax = axes[2]
for name, ret, color in [
    ("Walk-Forward", ret_wf, '#d4a24e'),
    ("Static Optimal", ret_static_oos, '#7c6dd8'),
    ("JPY Only", ret_jpy_oos, '#e05252'),
]:
    ret = ret.dropna()
    rs = ret.rolling(126).apply(lambda x: (x.mean()/x.std())*np.sqrt(252), raw=False)
    ax.plot(rs.index, rs.values, label=name, color=color, linewidth=1.2)

ax.axhline(0, color='white', linewidth=0.3, linestyle='--')
ax.set_title("Rolling 6M Sharpe Ratio", fontsize=12)
ax.set_ylabel("Sharpe")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# WEIGHT STABILITY ANALYSIS
# ============================================================
print("Walk-Forward Weight Statistics:")
print("(How stable are the optimal weights across re-optimization periods?)\n")
print(weights_history.describe().round(3))

print(f"\nJPY weight range:     {weights_history['JPY'].min():.1%} – {weights_history['JPY'].max():.1%}")
print(f"MXN_rev weight range: {weights_history['MXN_rev'].min():.1%} – {weights_history['MXN_rev'].max():.1%}")
print(f"CNY_rev weight range: {weights_history['CNY_rev'].min():.1%} – {weights_history['CNY_rev'].max():.1%}")

---
# Part 2: Value at Risk (VaR) & Conditional VaR (CVaR)

VaR answers: *"What's the maximum loss I should expect at a given confidence level?"*

CVaR (Expected Shortfall) answers: *"If losses exceed VaR, how bad does it get on average?"*

We compute both using three methods:
- **Historical:** Directly from the empirical return distribution
- **Parametric (Gaussian):** Assuming normal returns
- **Cornish-Fisher:** Adjusts for skewness and kurtosis (better for carry trades)

In [ ]:
# ============================================================
# VaR & CVaR CALCULATIONS
# ============================================================
def compute_var_cvar(returns, confidence_levels=[0.90, 0.95, 0.99]):
    """Compute VaR and CVaR using historical, parametric, and Cornish-Fisher methods."""
    mu = returns.mean()
    sigma = returns.std()
    skew = returns.skew()
    kurt = returns.kurtosis()  # excess kurtosis
    
    results = []
    
    for cl in confidence_levels:
        alpha = 1 - cl
        z = stats.norm.ppf(alpha)
        
        # --- Historical VaR ---
        hist_var = -np.percentile(returns, alpha * 100)
        hist_cvar = -returns[returns <= -hist_var].mean() if (returns <= -hist_var).sum() > 0 else hist_var
        
        # --- Parametric (Gaussian) VaR ---
        param_var = -(mu + z * sigma)
        # Gaussian CVaR: E[X | X < VaR] = mu - sigma * phi(z) / alpha
        param_cvar = -(mu - sigma * stats.norm.pdf(z) / alpha)
        
        # --- Cornish-Fisher VaR ---
        # Adjust z-score for skewness and kurtosis
        z_cf = (z + (z**2 - 1) * skew / 6
                  + (z**3 - 3*z) * kurt / 24
                  - (2*z**3 - 5*z) * skew**2 / 36)
        cf_var = -(mu + z_cf * sigma)
        cf_cvar = cf_var * 1.1  # approximate; CF doesn't have closed-form CVaR
        
        results.append({
            'Confidence': f"{cl:.0%}",
            'Hist VaR': hist_var,
            'Hist CVaR': hist_cvar,
            'Param VaR': param_var,
            'Param CVaR': param_cvar,
            'CF VaR': cf_var,
        })
    
    return pd.DataFrame(results).set_index('Confidence')

# Compute for both portfolios
print("=" * 70)
print("DAILY VaR & CVaR: OPTIMAL PORTFOLIO (70/27.4/2.6)")
print("=" * 70)
var_optimal = compute_var_cvar(ret_optimal)
display(var_optimal.map(lambda x: f"{x:.4%}" if isinstance(x, float) else x))

print(f"\nReturn distribution: mean={ret_optimal.mean():.5f}, std={ret_optimal.std():.5f}, skew={ret_optimal.skew():.3f}, kurt={ret_optimal.kurtosis():.2f}")

print("\n" + "=" * 70)
print("DAILY VaR & CVaR: JPY ONLY")
print("=" * 70)
var_jpy = compute_var_cvar(ret_jpy_only)
display(var_jpy.map(lambda x: f"{x:.4%}" if isinstance(x, float) else x))

print(f"\nReturn distribution: mean={ret_jpy_only.mean():.5f}, std={ret_jpy_only.std():.5f}, skew={ret_jpy_only.skew():.3f}, kurt={ret_jpy_only.kurtosis():.2f}")

In [ ]:
# ============================================================
# SCALE TO DIFFERENT HORIZONS
# ============================================================
# For a $100,000 portfolio
PORTFOLIO_VALUE = 100_000

horizons = {
    "1 Day": 1,
    "1 Week": 5,
    "1 Month": 21,
    "3 Months": 63,
}

print(f"VaR/CVaR for ${PORTFOLIO_VALUE:,.0f} Portfolio (95% confidence, Historical method)\n")
print(f"{'Horizon':<12} {'— Optimal Portfolio —':>30} {'— JPY Only —':>30}")
print(f"{'':12} {'VaR ($)':>14} {'CVaR ($)':>14} {'VaR ($)':>14} {'CVaR ($)':>14}")
print("-" * 74)

for label, days in horizons.items():
    # Scale daily VaR by sqrt(t) for parametric, or compute directly for historical
    if days == 1:
        opt_var = -np.percentile(ret_optimal, 5) * PORTFOLIO_VALUE
        opt_cvar = -ret_optimal[ret_optimal <= np.percentile(ret_optimal, 5)].mean() * PORTFOLIO_VALUE
        jpy_var = -np.percentile(ret_jpy_only, 5) * PORTFOLIO_VALUE
        jpy_cvar = -ret_jpy_only[ret_jpy_only <= np.percentile(ret_jpy_only, 5)].mean() * PORTFOLIO_VALUE
    else:
        # Compute multi-day returns by rolling
        opt_multi = ret_optimal.rolling(days).apply(lambda x: (1+x).prod()-1, raw=True).dropna()
        jpy_multi = ret_jpy_only.rolling(days).apply(lambda x: (1+x).prod()-1, raw=True).dropna()
        opt_var = -np.percentile(opt_multi, 5) * PORTFOLIO_VALUE
        opt_cvar = -opt_multi[opt_multi <= np.percentile(opt_multi, 5)].mean() * PORTFOLIO_VALUE
        jpy_var = -np.percentile(jpy_multi, 5) * PORTFOLIO_VALUE
        jpy_cvar = -jpy_multi[jpy_multi <= np.percentile(jpy_multi, 5)].mean() * PORTFOLIO_VALUE
    
    print(f"{label:<12} {opt_var:>13,.0f} {opt_cvar:>13,.0f} {jpy_var:>13,.0f} {jpy_cvar:>13,.0f}")

In [ ]:
# ============================================================
# VaR VISUALIZATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, ret, color) in zip(axes, [
    ("Optimal Portfolio", ret_optimal, '#d4a24e'),
    ("JPY Only", ret_jpy_only, '#e05252'),
]):
    ret = ret.dropna()
    
    # Histogram
    n, bins, patches = ax.hist(ret * 100, bins=100, density=True,
                               alpha=0.6, color=color, edgecolor='none')
    
    # VaR lines
    for cl, ls in [(0.95, '-'), (0.99, '--')]:
        var_val = np.percentile(ret, (1 - cl) * 100)
        ax.axvline(var_val * 100, color='white', linewidth=1.5, linestyle=ls,
                   label=f'{cl:.0%} VaR: {var_val:.3%}')
    
    # CVaR shading
    var_95 = np.percentile(ret, 5)
    ax.axvspan(ret.min() * 100, var_95 * 100, alpha=0.15, color='red',
               label=f'95% CVaR region')
    
    # Normal overlay
    x_range = np.linspace(ret.min() * 100, ret.max() * 100, 200)
    normal_pdf = stats.norm.pdf(x_range, ret.mean() * 100, ret.std() * 100)
    ax.plot(x_range, normal_pdf, 'w--', linewidth=1, alpha=0.5, label='Normal fit')
    
    ax.set_title(f"{name} — Daily Return Distribution", fontsize=12)
    ax.set_xlabel("Daily Return (%)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)
    ax.set_xlim(-3, 3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ROLLING VaR (95%) OVER TIME
# ============================================================
ROLL_WINDOW = 252  # 1 year

fig, ax = plt.subplots(figsize=(15, 6))

for name, ret, color in [
    ("Optimal Portfolio", ret_optimal, '#d4a24e'),
    ("JPY Only", ret_jpy_only, '#e05252'),
]:
    ret = ret.dropna()
    rolling_var = ret.rolling(ROLL_WINDOW).quantile(0.05) * PORTFOLIO_VALUE
    ax.plot(rolling_var.index, rolling_var.values, label=f"{name} 95% VaR",
            color=color, linewidth=1.5)

ax.set_title(f"Rolling 1-Year 95% Daily VaR (${PORTFOLIO_VALUE:,.0f} portfolio)", fontsize=13)
ax.set_ylabel("VaR ($ loss)")
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.invert_yaxis()  # losses shown as negative

plt.tight_layout()
plt.show()

---
# Part 3: Monte Carlo Simulation

Simulate thousands of potential future paths using three approaches:

1. **Parametric (Multivariate Normal):** Draw from fitted normal distribution
2. **Bootstrap (Block Resampling):** Preserve autocorrelation and fat tails
3. **Regime-Switching:** Separate calm/crisis distributions for more realistic tails

In [ ]:
# ============================================================
# MONTE CARLO: PARAMETRIC (MULTIVARIATE NORMAL)
# ============================================================
np.random.seed(42)

N_SIMS = 10_000
HORIZON = 252  # 1 year forward

# Fit multivariate normal to historical daily returns
mu_vec = returns_df.mean().values
cov_mat = returns_df.cov().values

# Simulate
terminal_returns_normal = []
paths_normal = []

for _ in range(N_SIMS):
    # Draw HORIZON days of correlated returns
    sim_returns = np.random.multivariate_normal(mu_vec, cov_mat, size=HORIZON)
    
    # Portfolio return for each day
    port_daily = sim_returns @ W_OPTIMAL
    
    # Cumulative path
    cum = np.cumprod(1 + port_daily)
    paths_normal.append(cum)
    terminal_returns_normal.append(cum[-1] - 1)

terminal_returns_normal = np.array(terminal_returns_normal)
paths_normal = np.array(paths_normal)

print(f"Parametric MC: {N_SIMS:,} simulations × {HORIZON} days")
print(f"\nTerminal Return Distribution (1 year):")
print(f"  Mean:   {terminal_returns_normal.mean():.2%}")
print(f"  Median: {np.median(terminal_returns_normal):.2%}")
print(f"  Std:    {terminal_returns_normal.std():.2%}")
print(f"  5th %%:  {np.percentile(terminal_returns_normal, 5):.2%}")
print(f"  1st %%:  {np.percentile(terminal_returns_normal, 1):.2%}")
print(f"  Worst:  {terminal_returns_normal.min():.2%}")
print(f"  Best:   {terminal_returns_normal.max():.2%}")
print(f"  P(loss): {(terminal_returns_normal < 0).mean():.1%}")

In [ ]:
# ============================================================
# MONTE CARLO: BLOCK BOOTSTRAP
# ============================================================
# Resample blocks of actual historical returns to preserve
# autocorrelation, fat tails, and cross-asset dependence structure

BLOCK_SIZE = 21  # ~1 month blocks
n_blocks = HORIZON // BLOCK_SIZE + 1

terminal_returns_bootstrap = []
paths_bootstrap = []
portfolio_daily = (returns_df * W_OPTIMAL).sum(axis=1).values

for _ in range(N_SIMS):
    # Randomly select block starting points
    max_start = len(portfolio_daily) - BLOCK_SIZE
    starts = np.random.randint(0, max_start, size=n_blocks)
    
    # Stitch blocks together
    sim_daily = np.concatenate([portfolio_daily[s:s+BLOCK_SIZE] for s in starts])[:HORIZON]
    
    cum = np.cumprod(1 + sim_daily)
    paths_bootstrap.append(cum)
    terminal_returns_bootstrap.append(cum[-1] - 1)

terminal_returns_bootstrap = np.array(terminal_returns_bootstrap)
paths_bootstrap = np.array(paths_bootstrap)

print(f"Block Bootstrap MC: {N_SIMS:,} simulations × {HORIZON} days (block size={BLOCK_SIZE})")
print(f"\nTerminal Return Distribution (1 year):")
print(f"  Mean:   {terminal_returns_bootstrap.mean():.2%}")
print(f"  Median: {np.median(terminal_returns_bootstrap):.2%}")
print(f"  Std:    {terminal_returns_bootstrap.std():.2%}")
print(f"  5th %%:  {np.percentile(terminal_returns_bootstrap, 5):.2%}")
print(f"  1st %%:  {np.percentile(terminal_returns_bootstrap, 1):.2%}")
print(f"  Worst:  {terminal_returns_bootstrap.min():.2%}")
print(f"  Best:   {terminal_returns_bootstrap.max():.2%}")
print(f"  P(loss): {(terminal_returns_bootstrap < 0).mean():.1%}")

In [ ]:
# ============================================================
# MONTE CARLO: REGIME-SWITCHING
# ============================================================
# Estimate two regimes based on realized volatility:
# "Calm" (below-median vol) and "Crisis" (above-median vol)

# Classify historical days into regimes
rolling_vol = ret_optimal.rolling(21).std()
vol_median = rolling_vol.median()

calm_mask = rolling_vol <= vol_median
crisis_mask = rolling_vol > vol_median

# Fit distributions for each regime
port_daily_series = ret_optimal.dropna()

calm_returns = port_daily_series[calm_mask.reindex(port_daily_series.index).fillna(False)].dropna()
crisis_returns = port_daily_series[crisis_mask.reindex(port_daily_series.index).fillna(False)].dropna()

mu_calm, sig_calm = calm_returns.mean(), calm_returns.std()
mu_crisis, sig_crisis = crisis_returns.mean(), crisis_returns.std()

# Transition probabilities (from historical data)
calm_days = calm_mask.reindex(port_daily_series.index).fillna(False)
# P(crisis tomorrow | calm today)
transitions = calm_days.astype(int).diff().dropna()
p_calm_to_crisis = (transitions == -1).sum() / (calm_days[:-1]).sum()
p_crisis_to_calm = (transitions == 1).sum() / (~calm_days[:-1]).sum()

print(f"Regime Statistics:")
print(f"  Calm   — mean: {mu_calm*252:.2%} ann., vol: {sig_calm*np.sqrt(252):.2%} ann., {len(calm_returns)} days ({len(calm_returns)/len(port_daily_series):.0%})")
print(f"  Crisis — mean: {mu_crisis*252:.2%} ann., vol: {sig_crisis*np.sqrt(252):.2%} ann., {len(crisis_returns)} days ({len(crisis_returns)/len(port_daily_series):.0%})")
print(f"  P(calm→crisis): {p_calm_to_crisis:.3f}")
print(f"  P(crisis→calm): {p_crisis_to_calm:.3f}")

# Simulate
terminal_returns_regime = []
paths_regime = []

for _ in range(N_SIMS):
    sim_daily = np.zeros(HORIZON)
    in_crisis = np.random.random() < 0.5  # random initial state
    
    for t in range(HORIZON):
        if in_crisis:
            sim_daily[t] = np.random.normal(mu_crisis, sig_crisis)
            if np.random.random() < p_crisis_to_calm:
                in_crisis = False
        else:
            sim_daily[t] = np.random.normal(mu_calm, sig_calm)
            if np.random.random() < p_calm_to_crisis:
                in_crisis = True
    
    cum = np.cumprod(1 + sim_daily)
    paths_regime.append(cum)
    terminal_returns_regime.append(cum[-1] - 1)

terminal_returns_regime = np.array(terminal_returns_regime)
paths_regime = np.array(paths_regime)

print(f"\nRegime-Switching MC: {N_SIMS:,} simulations × {HORIZON} days")
print(f"\nTerminal Return Distribution (1 year):")
print(f"  Mean:   {terminal_returns_regime.mean():.2%}")
print(f"  Median: {np.median(terminal_returns_regime):.2%}")
print(f"  Std:    {terminal_returns_regime.std():.2%}")
print(f"  5th %%:  {np.percentile(terminal_returns_regime, 5):.2%}")
print(f"  1st %%:  {np.percentile(terminal_returns_regime, 1):.2%}")
print(f"  Worst:  {terminal_returns_regime.min():.2%}")
print(f"  Best:   {terminal_returns_regime.max():.2%}")
print(f"  P(loss): {(terminal_returns_regime < 0).mean():.1%}")

In [ ]:
# ============================================================
# MC VISUALIZATION: FAN CHARTS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

days = np.arange(HORIZON)
percentile_bands = [(5, 95), (10, 90), (25, 75)]
alphas = [0.15, 0.25, 0.4]

for ax, (title, paths, color) in zip(axes, [
    ("Parametric (Normal)", paths_normal, '#d4a24e'),
    ("Block Bootstrap", paths_bootstrap, '#3ba1c9'),
    ("Regime-Switching", paths_regime, '#7c6dd8'),
]):
    # Percentile bands
    for (lo, hi), alpha in zip(percentile_bands, alphas):
        p_lo = np.percentile(paths, lo, axis=0)
        p_hi = np.percentile(paths, hi, axis=0)
        ax.fill_between(days, p_lo, p_hi, alpha=alpha, color=color,
                        label=f'{lo}th–{hi}th pctile' if alpha == 0.15 else None)
    
    # Median
    median_path = np.median(paths, axis=0)
    ax.plot(days, median_path, color=color, linewidth=2, label='Median')
    
    # A few sample paths
    for i in range(20):
        ax.plot(days, paths[i], color=color, alpha=0.05, linewidth=0.5)
    
    ax.axhline(1, color='white', linewidth=0.3, linestyle='--')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Trading Days")
    ax.set_ylabel("Growth of $1")
    ax.legend(fontsize=8)
    ax.set_ylim(0.7, 1.4)

plt.suptitle(f"Monte Carlo Fan Charts: {N_SIMS:,} Simulated 1-Year Paths", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TERMINAL RETURN DISTRIBUTIONS: ALL THREE METHODS
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

for name, terminal, color in [
    ("Parametric (Normal)", terminal_returns_normal, '#d4a24e'),
    ("Block Bootstrap", terminal_returns_bootstrap, '#3ba1c9'),
    ("Regime-Switching", terminal_returns_regime, '#7c6dd8'),
]:
    ax.hist(terminal * 100, bins=100, density=True, alpha=0.5,
            color=color, label=name, edgecolor='none')
    
    # 5th percentile line
    p5 = np.percentile(terminal, 5)
    ax.axvline(p5 * 100, color=color, linewidth=1.5, linestyle='--', alpha=0.8)

ax.axvline(0, color='white', linewidth=0.5, linestyle='-')
ax.set_title("1-Year Terminal Return Distribution (All MC Methods)", fontsize=13)
ax.set_xlabel("1-Year Return (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPREHENSIVE SUMMARY TABLE
# ============================================================
print("=" * 80)
print("MONTE CARLO SUMMARY: 1-YEAR HORIZON, OPTIMAL PORTFOLIO")
print("=" * 80)
print(f"{'Metric':<28} {'Parametric':>14} {'Bootstrap':>14} {'Regime-SW':>14}")
print("-" * 80)

for label, func in [
    ("Mean return", lambda x: f"{x.mean():.2%}"),
    ("Median return", lambda x: f"{np.median(x):.2%}"),
    ("Std deviation", lambda x: f"{x.std():.2%}"),
    ("Skewness", lambda x: f"{stats.skew(x):.3f}"),
    ("Kurtosis (excess)", lambda x: f"{stats.kurtosis(x):.2f}"),
    ("P(loss)", lambda x: f"{(x < 0).mean():.1%}"),
    ("P(loss > 10%)", lambda x: f"{(x < -0.10).mean():.2%}"),
    ("P(loss > 20%)", lambda x: f"{(x < -0.20).mean():.2%}"),
    ("95% VaR", lambda x: f"{-np.percentile(x, 5):.2%}"),
    ("99% VaR", lambda x: f"{-np.percentile(x, 1):.2%}"),
    ("95% CVaR", lambda x: f"{-x[x <= np.percentile(x, 5)].mean():.2%}"),
    ("99% CVaR", lambda x: f"{-x[x <= np.percentile(x, 1)].mean():.2%}"),
    ("Best case", lambda x: f"{x.max():.2%}"),
    ("Worst case", lambda x: f"{x.min():.2%}"),
]:
    vals = [func(terminal_returns_normal), func(terminal_returns_bootstrap), func(terminal_returns_regime)]
    print(f"{label:<28} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14}")

In [ ]:
# ============================================================
# COMPARE: OPTIMAL PORTFOLIO vs JPY ONLY (MC)
# ============================================================
# Run bootstrap MC for JPY Only
jpy_daily = ret_jpy_only.dropna().values

terminal_jpy_bootstrap = []
for _ in range(N_SIMS):
    max_start = len(jpy_daily) - BLOCK_SIZE
    starts = np.random.randint(0, max_start, size=n_blocks)
    sim = np.concatenate([jpy_daily[s:s+BLOCK_SIZE] for s in starts])[:HORIZON]
    terminal_jpy_bootstrap.append(np.prod(1 + sim) - 1)

terminal_jpy_bootstrap = np.array(terminal_jpy_bootstrap)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Side-by-side histograms
for ax, (name, terminal, color) in zip(axes, [
    ("Optimal Portfolio (Bootstrap)", terminal_returns_bootstrap, '#d4a24e'),
    ("JPY Only (Bootstrap)", terminal_jpy_bootstrap, '#e05252'),
]):
    ax.hist(terminal * 100, bins=80, density=True, alpha=0.7,
            color=color, edgecolor='none')
    
    # Key stats as text
    textstr = (f"Mean: {terminal.mean():.1%}\n"
               f"Median: {np.median(terminal):.1%}\n"
               f"P(loss): {(terminal<0).mean():.0%}\n"
               f"95% VaR: {-np.percentile(terminal, 5):.1%}\n"
               f"Worst: {terminal.min():.1%}")
    ax.text(0.97, 0.95, textstr, transform=ax.transAxes,
            verticalalignment='top', horizontalalignment='right',
            fontsize=10, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    
    ax.axvline(0, color='white', linewidth=0.5)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel("1-Year Return (%)")
    ax.set_ylabel("Density")

plt.suptitle("Monte Carlo: Optimal vs JPY Only (Block Bootstrap, 10K sims)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Comparative stats
print(f"\n{'Metric':<28} {'Optimal':>14} {'JPY Only':>14} {'Difference':>14}")
print("-" * 72)
for label, func in [
    ("Mean return", lambda x: x.mean()),
    ("Std deviation", lambda x: x.std()),
    ("P(loss)", lambda x: (x < 0).mean()),
    ("95% VaR", lambda x: -np.percentile(x, 5)),
    ("99% VaR", lambda x: -np.percentile(x, 1)),
    ("95% CVaR", lambda x: -x[x <= np.percentile(x, 5)].mean()),
]:
    v_opt = func(terminal_returns_bootstrap)
    v_jpy = func(terminal_jpy_bootstrap)
    diff = v_opt - v_jpy
    print(f"{label:<28} {v_opt:>13.2%} {v_jpy:>13.2%} {diff:>+13.2%}")

---
# Part 4: Confidence Intervals on Maximum Loss

Using the MC simulations, we can build confidence intervals around the max drawdown
and worst-case loss at various holding periods.

In [ ]:
# ============================================================
# MAX DRAWDOWN DISTRIBUTION FROM MC
# ============================================================
def compute_max_drawdown(cum_path):
    """Compute max drawdown from a cumulative return path."""
    peak = np.maximum.accumulate(cum_path)
    dd = (cum_path - peak) / peak
    return dd.min()

# Compute max DD for each bootstrap simulation
max_dds_optimal = np.array([compute_max_drawdown(p) for p in paths_bootstrap])

# Also for JPY only
paths_jpy_bootstrap = []
for _ in range(N_SIMS):
    max_start = len(jpy_daily) - BLOCK_SIZE
    starts = np.random.randint(0, max_start, size=n_blocks)
    sim = np.concatenate([jpy_daily[s:s+BLOCK_SIZE] for s in starts])[:HORIZON]
    paths_jpy_bootstrap.append(np.cumprod(1 + sim))

max_dds_jpy = np.array([compute_max_drawdown(p) for p in paths_jpy_bootstrap])

fig, ax = plt.subplots(figsize=(14, 6))

ax.hist(max_dds_optimal * 100, bins=80, density=True, alpha=0.6,
        color='#d4a24e', label='Optimal Portfolio', edgecolor='none')
ax.hist(max_dds_jpy * 100, bins=80, density=True, alpha=0.6,
        color='#e05252', label='JPY Only', edgecolor='none')

# Confidence intervals
for pct, ls in [(50, '-'), (95, '--'), (99, ':')]:
    v_opt = np.percentile(max_dds_optimal, pct)
    v_jpy = np.percentile(max_dds_jpy, pct)
    ax.axvline(v_opt * 100, color='#d4a24e', linestyle=ls, linewidth=1.5, alpha=0.8)
    ax.axvline(v_jpy * 100, color='#e05252', linestyle=ls, linewidth=1.5, alpha=0.8)

ax.set_title("Max Drawdown Distribution (1-Year Horizon, Bootstrap MC)", fontsize=13)
ax.set_xlabel("Max Drawdown (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nMax Drawdown Confidence Intervals (1-Year):")
print(f"\n{'Percentile':<16} {'Optimal':>14} {'JPY Only':>14}")
print("-" * 46)
for pct in [50, 75, 90, 95, 99]:
    v_opt = np.percentile(max_dds_optimal, pct)
    v_jpy = np.percentile(max_dds_jpy, pct)
    print(f"{pct}th percentile  {v_opt:>13.2%} {v_jpy:>13.2%}")

print(f"\nInterpretation: There is a 95% probability that the max drawdown")
print(f"over the next year will be no worse than {np.percentile(max_dds_optimal, 95):.2%} (optimal)")
print(f"vs {np.percentile(max_dds_jpy, 95):.2%} (JPY only).")

In [ ]:
# ============================================================
# LOSS PROBABILITY CURVES
# ============================================================
loss_thresholds = np.arange(0, 0.35, 0.005)

fig, ax = plt.subplots(figsize=(14, 6))

for name, terminal, color, ls in [
    ("Optimal (Bootstrap)", terminal_returns_bootstrap, '#d4a24e', '-'),
    ("Optimal (Regime-SW)", terminal_returns_regime, '#7c6dd8', '--'),
    ("JPY Only (Bootstrap)", terminal_jpy_bootstrap, '#e05252', '-'),
]:
    probs = [(terminal < -t).mean() for t in loss_thresholds]
    ax.plot(loss_thresholds * 100, np.array(probs) * 100,
            color=color, linewidth=2, linestyle=ls, label=name)

ax.set_title("Probability of Exceeding Loss Threshold (1-Year Horizon)", fontsize=13)
ax.set_xlabel("Loss Threshold (%)")
ax.set_ylabel("Probability (%)")
ax.legend(fontsize=10)
ax.set_xlim(0, 30)
ax.set_ylim(0, 50)

# Reference lines
for loss_pct in [5, 10, 15, 20]:
    ax.axvline(loss_pct, color='white', linewidth=0.3, linestyle=':')

plt.tight_layout()
plt.show()

print("\nProbability of exceeding loss thresholds (1-year):")
print(f"\n{'Loss >':<12} {'Optimal (BS)':>14} {'Optimal (RS)':>14} {'JPY Only':>14}")
print("-" * 58)
for t in [0.05, 0.10, 0.15, 0.20, 0.25]:
    p_opt_bs = (terminal_returns_bootstrap < -t).mean()
    p_opt_rs = (terminal_returns_regime < -t).mean()
    p_jpy = (terminal_jpy_bootstrap < -t).mean()
    print(f"{t:>10.0%}   {p_opt_bs:>13.2%} {p_opt_rs:>13.2%} {p_jpy:>13.2%}")

## Summary

### Walk-Forward
Compare the walk-forward Sharpe to the in-sample 0.864. If it's close (say >0.7), the allocation
is robust. Check whether the weights are stable over time or swing wildly.

### VaR / CVaR
The key numbers for risk management — at 95% confidence, how much can you lose in a day, week,
or month? CVaR matters more than VaR for carry trades due to the fat left tail.

### Monte Carlo
The bootstrap and regime-switching models should give wider tails than the parametric model.
If the bootstrap shows meaningfully worse tail outcomes than the normal MC, it confirms that
the Gaussian assumption understates carry trade risk. The regime-switching model is the most
conservative and likely the most realistic for carry strategies.

Key things to watch:
- Is P(loss) substantially lower for the optimal portfolio vs JPY only?
- Are the 95% and 99% VaR/CVaR numbers meaningfully reduced?
- Does the max drawdown confidence interval tighten?